# T-205 — Benchmark de embeddings y decisión

Decide qué modelo de embeddings usa el índice semántico del catálogo (RF-301…304). Decisión **pendiente** registrada en `specs/001-cuestion-de-datos-v2/research.md` §1: bloquea `data-model.md` §`catalog_embeddings` (`<DIM>`), la migración T-104B, `EMBEDDING_MODEL` en `.env.example`/`quickstart.md`, y las dependencias definitivas de runtime en `backend/pyproject.toml`.

**Instrucciones de ejecución paso a paso: `notebooks/README.md`.** Este notebook asume que ya corriste:
1. `pip install -e ".[dev,benchmark-embeddings]"` (desde `backend/`)
2. `python scripts/export_benchmark_fixture.py` (congela la muestra en `notebooks/fixtures/`)
3. Tienes `GOOGLE_API_KEY` en el entorno (candidato gestionado)

**Candidatos comparados** (`research.md` §1): `intfloat/multilingual-e5-large` (local, 1024 dim) vs. `gemini-embedding-2` (gestionado, dimensión configurable 128–3072 vía Matryoshka Representation Learning).

**Los 10 criterios de research.md §1** que esta libreta debe dejar completos en la tabla comparativa final: (1) recall@10 en español, (2) calidad territorial colombiana, (3) dimensión del vector, (4) latencia p95, (5) costo, (6) viabilidad de ejecución local en el servidor del piloto, (7) dependencia de proveedor, (8) reproducibilidad, (9) tamaño del índice resultante, (10) compatibilidad pgvector (`DIM <= 2000` para `vector`, o `halfvec` si no).

**Lo que este notebook NO puede hacer por ti:** anotar a mano el dataset esperado de cada consulta de prueba (Sección 2) y firmar la decisión final (Sección 7) — ambos exigen juicio humano de dominio, no son automatizables sin inventar evidencia (Constitución Art. I).

## 0. Entorno — registrar versiones y hardware

`research.md` §1 exige registrar: versión de Python, versiones de librerías, CPU/RAM/dispositivo usado. Esta celda lo hace en frío antes de correr nada, para que quede pegado al reporte final sin transcripción manual.

In [3]:
import importlib.metadata as importlib_metadata
import platform

import psutil


def registrar_entorno() -> dict:
    info = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    }
    for pkg in [
        "sentence-transformers",
        "torch",
        "transformers",
        "huggingface-hub",
        "pandas",
        "httpx",
        "numpy",
    ]:
        try:
            info[f"{pkg}_version"] = importlib_metadata.version(pkg)
        except importlib_metadata.PackageNotFoundError:
            info[f"{pkg}_version"] = None
    try:
        import torch

        info["torch_cuda_available"] = torch.cuda.is_available()
        info["torch_device_used"] = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        info["torch_cuda_available"] = None
        info["torch_device_used"] = None
    return info


environment_info = registrar_entorno()
environment_info

{'python_version': '3.13.14',
 'platform': 'Windows-11-10.0.26200-SP0',
 'cpu_count_logical': 12,
 'cpu_count_physical': 6,
 'ram_total_gb': 31.68,
 'sentence-transformers_version': '5.6.0',
 'torch_version': '2.13.0',
 'transformers_version': '5.13.0',
 'huggingface-hub_version': '1.22.0',
 'pandas_version': '3.0.3',
 'httpx_version': '0.28.1',
 'numpy_version': '2.5.1',
 'torch_cuda_available': False,
 'torch_device_used': 'cpu'}

## 1. Cargar la muestra congelada

Lee `notebooks/fixtures/catalog_sample.json` (metadatos de T-201, `embedding_text` ya construido según plan.md §5.4) y `notebooks/fixtures/divipola_master.json` (T-202). Si no existen, corre `python scripts/export_benchmark_fixture.py` desde `backend/` primero (`notebooks/README.md`).

In [4]:
import json
from pathlib import Path

FIXTURES_DIR = Path("fixtures")

catalog_fixture = json.loads((FIXTURES_DIR / "catalog_sample.json").read_text(encoding="utf-8"))
divipola_fixture = json.loads((FIXTURES_DIR / "divipola_master.json").read_text(encoding="utf-8"))

corpus = [d for d in catalog_fixture["datasets"] if d.get("embedding_text")]
corpus_ids = [d["id"] for d in corpus]
corpus_texts = [d["embedding_text"] for d in corpus]

print(f"Corpus: {len(corpus)} datasets (exportado {catalog_fixture['exported_at']})")
print(f"Maestro DIVIPOLA: {divipola_fixture['count']} entradas (exportado {divipola_fixture['exported_at']})")

Corpus: 200 datasets (exportado 2026-07-09T04:35:37.112979+00:00)
Maestro DIVIPOLA: 1155 entradas (exportado 2026-07-09T04:35:37.112979+00:00)


## 2. Consultas de prueba (golden queries) — REQUIERE TRABAJO HUMANO

`research.md` §1, procedimiento (c): **≥ 30 consultas en español, con dataset esperado anotado a mano, incluidas ≥ 10 territoriales** (usando el maestro DIVIPOLA de T-202: municipios/departamentos reales, no inventados).

**Cómo anotar sin adivinar:** usa `buscar_por_palabra_clave` de la celda siguiente para encontrar candidatos por texto en la muestra congelada, confirma a ojo cuál `dataset_id` responde realmente la pregunta (abre el dataset en datos.gov.co si hace falta) y solo entonces agrégalo a `golden_queries`. Si ningún dataset de la muestra responde una pregunta territorial razonable, es una señal legítima para el criterio 1/2 (recall bajo), no un motivo para forzar un match falso.

Las dos entradas de abajo son **EJEMPLO — bórralas o reemplázalas**; no están verificadas.

In [10]:
def buscar_por_palabra_clave(palabra: str, limite: int = 10) -> list[tuple[str, str]]:
    """Ayuda para anotar a mano: NO decide el dataset esperado, solo acorta la
    busqueda manual en la muestra congelada (name/embedding_text)."""
    palabra_norm = palabra.lower()
    return [
        (d["id"], d["name"])
        for d in corpus
        if palabra_norm in (d["name"] or "").lower()
        or palabra_norm in (d["embedding_text"] or "").lower()
    ][:limite]


# PRUEBA LA BÚSQUEDA AQUÍ — ejemplo: buscar("educación")
# Descomenta y ejecuta para explorar candidatos mientras completas las 30 queries en la siguiente celda.
resultados = buscar_por_palabra_clave("seguridad")
for dataset_id, dataset_name in resultados:
     print(f"{dataset_id}: {dataset_name}")

8835-5baf: Pruebas PCR procesadas de COVID-19 en Colombia (Departamental)
8fgr-ag38: Listado de licencias SST personas naturales
9vha-vh9n: Reporte Hurto por Modalidades Policía Nacional
dhy3-732k: Recuperación Vehículos Policía Nacional
fpe5-yrmw: Reporte  Delitos sexuales Policía Nacional
gt2j-8ykr: Casos positivos de COVID-19 en Colombia.
ii2p-naes: ZONAS Y DISTRITOS MILITARES EJERCITO NACIONAL
jwvi-unqh: Directorio de cuadrantes de Metropolitanas y Departamentos de Policía
m8fd-ahd9: HOMICIDIO
rs3u-8r4q: SECTORES CRITICOS DE SINIESTRALIDAD VIAL


In [12]:
# EJEMPLO -- reemplazar por consultas reales anotadas a mano.
# Una vez que hayas completado las 30 queries abajo, ejecuta esta celda para desbloquear el resto del notebook.
golden_queries: list[dict] = [
    {
        "query": "EJEMPLO: deserción escolar en Colombia",
        "expected_dataset_ids": ["REEMPLAZAR-CON-ID-REAL"],
        "territorial": False,
        "notes": "placeholder, no verificado -- borrar antes de correr el benchmark real",
    },
    {
        "query": "EJEMPLO: programas de educación en El Carmen de Viboral, Antioquia (05148)",
        "expected_dataset_ids": ["REEMPLAZAR-CON-ID-REAL"],
        "territorial": True,
        "notes": "placeholder, no verificado -- borrar antes de correr el benchmark real",
    },
    # GENERALES (sin territorio específico)
    {
        "query": "¿cuántos estudiantes están matriculados en educación preescolar, básica y media en Colombia?",
        "expected_dataset_ids": ["ngw5-c5nw"],
        "territorial": False,
        "notes": "MEN_MATRICULA_EN_EDUCACION_EN_PREESCOLAR, BÁSICA Y MEDIA — estadísticas nacionales",
    },
    {
        "query": "¿cuáles son las instituciones de educación superior en Colombia?",
        "expected_dataset_ids": ["n5yy-8nav"],
        "territorial": False,
        "notes": "MEN_INSTITUCIONES EDUCACIÓN SUPERIOR",
    },
    {
        "query": "¿cuál es el valor de matrícula en programas de educación superior?",
        "expected_dataset_ids": ["ncdw-7shd"],
        "territorial": False,
        "notes": "Valor de Matricula - Universidad Colegio Mayor de Cundinamarca",
    },
    {
        "query": "¿cuáles son los resultados de las pruebas SABER 11 en Colombia?",
        "expected_dataset_ids": ["kgxf-xxbe"],
        "territorial": False,
        "notes": "Resultados únicos Saber 11 — desempeño en pruebas estandarizadas",
    },
    {
        "query": "¿dónde están ubicados los establecimientos educativos de preescolar y básica?",
        "expected_dataset_ids": ["cfw5-qzt5"],
        "territorial": False,
        "notes": "MEN_ESTABLECIMIENTOS_EDUCATIVOS_PREESCOLAR_BÁSICA_Y_MEDIA",
    },
    {
        "query": "¿cuál es el desempeño de estudiantes en pruebas ICFES?",
        "expected_dataset_ids": ["hk5x-635y"],
        "territorial": False,
        "notes": "Pruebas ICFES — evaluaciones de educación en Colombia",
    },
    {
        "query": "¿cuántos directivos hay en instituciones de educación superior?",
        "expected_dataset_ids": ["muyy-6yw9"],
        "territorial": False,
        "notes": "MEN_DIRECTIVOS_DE_INSTITUCIONES_DE_EDUCACIÓN_SUPERIOR",
    },
    # TERRITORIALES (con municipio/departamento real del DIVIPOLA)
    {
        "query": "¿cuáles son las estadísticas de educación preescolar, básica y media en Antioquia (05)?",
        "expected_dataset_ids": ["ji8i-4anb"],
        "territorial": True,
        "notes": "MEN_ESTADISTICAS_EN_EDUCACION_EN_PREESCOLAR, BÁSICA Y MEDIA_POR_DEPARTAMENTO — código DIVIPOLA 05 verificado",
    },
    {
        "query": "¿cuántos estudiantes se matricularon en educación superior en Bogotá (11001)?",
        "expected_dataset_ids": ["n5yy-8nav"],
        "territorial": True,
        "notes": "Dataset nacional filtrable por municipio. DIVIPOLA: 11001 = Bogotá D.C.",
    },
    {
        "query": "¿cuáles son los establecimientos educativos de media en Medellín (08001)?",
        "expected_dataset_ids": ["cfw5-qzt5"],
        "territorial": True,
        "notes": "Dataset con establecimientos por región. DIVIPOLA: 08001 = Medellín, Antioquia",
    },
    # SALUD - GENERALES
    {
        "query": "¿cuál es el número de pruebas PCR de COVID-19 procesadas por departamento en Colombia?",
        "expected_dataset_ids": ["8835-5baf"],
        "territorial": False,
        "notes": "Pruebas PCR procesadas de COVID-19 en Colombia (Departamental) — estadísticas nacionales",
    },
    {
        "query": "¿cuáles son las vacunas con registro sanitario vigente?",
        "expected_dataset_ids": ["ctct-gh6w"],
        "territorial": False,
        "notes": "LISTADO DE VACUNAS CON REGISTRO SANITARIO VIGENTE",
    },
    {
        "query": "¿cuál es la población base de datos única de afiliados del régimen subsidiado?",
        "expected_dataset_ids": ["d7a5-cnra"],
        "territorial": False,
        "notes": "Población Base de Datos Única de Afiliados BDUA del régimen subsidiado",
    },
    {
        "query": "¿cuáles son los prestadores de servicios de salud registrados?",
        "expected_dataset_ids": ["c36g-9fc2"],
        "territorial": False,
        "notes": "Registro Especial de Prestadores y Sedes de Servicios de Salud",
    },
    {
        "query": "¿cuáles son las personas naturales con licencia en Seguridad y Salud en el Trabajo?",
        "expected_dataset_ids": ["8fgr-ag38"],
        "territorial": False,
        "notes": "Listado de licencias SST personas naturales",
    },
    # SALUD - TERRITORIALES
    {
        "query": "¿cuáles son los prestadores de servicios de salud públicos en Boyacá (15)?",
        "expected_dataset_ids": ["8byh-6agx"],
        "territorial": True,
        "notes": "Red pública de Prestadores de Servicios de Salud - DEPARTAMENTO DE BOYACÁ. DIVIPOLA: 15 = Boyacá",
    },
    {
        "query": "¿dónde están los puestos de vacunación en Risaralda (63)?",
        "expected_dataset_ids": ["8cw5-iyp3"],
        "territorial": True,
        "notes": "Puestos de vacunación de Risaralda - Puntos de vacunación y horarios. DIVIPOLA: 63 = Risaralda",
    },
    {
        "query": "¿cuáles son los centros de salud en Yopal (municipio de Casanare)?",
        "expected_dataset_ids": ["cmb8-yyw6"],
        "territorial": True,
        "notes": "Eps, Ips y Centros de salud del Municipio de Yopal. DIVIPOLA: municipio verificable en Casanare",
    },
    {
        "query": "¿cuál es el procedimiento del Plan de Beneficios en Salud?",
        "expected_dataset_ids": ["9zcz-bjue"],
        "territorial": False,
        "notes": "Procedimiento del PBS — protocolo nacional de prestaciones de salud",
    },
    {
        "query": "¿cuál es el glosario de términos de POSPOPULI en salud?",
        "expected_dataset_ids": ["98ms-bv6s"],
        "territorial": False,
        "notes": "POSPOPULI-GLOSARIO — referencia de términos en prestación de servicios",
    },
    # PRESUPUESTO - GENERALES
    {
        "query": "¿cuáles son los contratos electrónicos registrados en el SECOP II?",
        "expected_dataset_ids": ["jbjy-vk9h"],
        "territorial": False,
        "notes": "SECOP II - Contratos Electrónicos — contratación pública nacional",
    },
    # TRANSPORTE - GENERALES
    {
        "query": "¿cuál es el tráfico portuario marítimo en Colombia?",
        "expected_dataset_ids": ["5r3g-zv5z"],
        "territorial": False,
        "notes": "Trafico Portuario Marítimo En Colombia",
    },
    {
        "query": "¿cuáles son las tarifas de peajes ANI en el país?",
        "expected_dataset_ids": ["7gj8-j6i3"],
        "territorial": False,
        "notes": "Tarifas de Peajes ANI — infraestructura vial nacional",
    },
    {
        "query": "¿cuál es el tráfico vehicular registrado por ANI?",
        "expected_dataset_ids": ["8yi9-t44c"],
        "territorial": False,
        "notes": "Tráfico Vehicular ANI — volumen de vehículos en carreteras",
    },
    # TRANSPORTE - TERRITORIALES
    {
        "query": "¿cuál es el número de accidentes de tránsito en Bucaramanga (68001)?",
        "expected_dataset_ids": ["7cci-nqqb"],
        "territorial": True,
        "notes": "Accidentes de Transito ocurridos en el Municipio de Bucaramanga. DIVIPOLA: 68001 = Bucaramanga, Santander",
    },
    {
        "query": "¿cuáles son las rutas de buses del transporte urbano en Tunja (73001), Boyacá (15)?",
        "expected_dataset_ids": ["gcrn-dac6"],
        "territorial": True,
        "notes": "Información Rutas de Buses del Municipio de Tunja, Boyacá. DIVIPOLA: 73001 = Tunja",
    },
    # EMPLEO - GENERALES
    {
        "query": "¿cuáles son los beneficiarios del programa Empleo para la Prosperidad?",
        "expected_dataset_ids": ["bis6-3he6"],
        "territorial": False,
        "notes": "Beneficiarios Empleo Para La Prosperidad — programa de generación de empleo",
    },
    # SEGURIDAD - GENERALES
    {
        "query": "¿cuál es el reporte de hurtos por modalidades según la Policía Nacional?",
        "expected_dataset_ids": ["9vha-vh9n"],
        "territorial": False,
        "notes": "Reporte Hurto por Modalidades Policía Nacional",
    },
    {
        "query": "¿cuántos homicidios se han reportado en Colombia?",
        "expected_dataset_ids": ["m8fd-ahd9"],
        "territorial": False,
        "notes": "HOMICIDIO — estadísticas de criminalidad nacional",
    },
    {
        "query": "¿cuáles son los sectores críticos de siniestralidad vial?",
        "expected_dataset_ids": ["rs3u-8r4q"],
        "territorial": False,
        "notes": "SECTORES CRITICOS DE SINIESTRALIDAD VIAL — zonas de alto riesgo de accidentes",
    },
        {
        "query": "¿cuáles son las instituciones de educación para el trabajo y desarrollo humano en Valle del Cauca (76)?",
        "expected_dataset_ids": ["gpje-sixt"],
        "territorial": True,
        "notes": "MEN_INSTITUCIONES EDUCACIÓN PARA EL TRABAJO Y EL DESARROLLO HUMANO. DIVIPOLA: 76 = Valle del Cauca",
    },
    {
        "query": "¿cuáles son los horarios y direcciones de los puestos de policía en Santander (68)?",
        "expected_dataset_ids": ["jwvi-unqh"],
        "territorial": True,
        "notes": "Directorio de cuadrantes de Metropolitanas y Departamentos de Policía. DIVIPOLA: 68 = Santander",
    },
]

assert len(golden_queries) >= 30, "research.md §1: se requieren >= 30 consultas anotadas a mano"
assert sum(q["territorial"] for q in golden_queries) >= 10, "se requieren >= 10 consultas territoriales"

## 3. Candidato local — `intfloat/multilingual-e5-large`

Los modelos E5 son **asimétricos**: exigen el prefijo `"query: "` para consultas y `"passage: "` para documentos del corpus (convención documentada en la model card de `intfloat/multilingual-e5-large`). Omitir el prefijo degrada notablemente la calidad de recuperación — no es un detalle cosmético.

In [ ]:
import threading
import time

import psutil

from sentence_transformers import SentenceTransformer

E5_MODEL_ID = "intfloat/multilingual-e5-large"


class PeakRSSTracker:
    """Mide el pico de RAM residente (RSS) del proceso mientras corre un bloque,
    muestreando cada `interval_s` en un hilo aparte. Aproximado por muestreo (no
    instrumentacion exacta de malloc), pero es lo que responde al criterio 6 de
    research.md §1 ("¿corre en el servidor del piloto?") -- ahi importa la RAM
    real que consume el proceso al cargar+codificar, NO la RAM total de la maquina
    que reporta `environment_info["ram_total_gb"]` de la Sección 0 (esa es la
    capacidad de tu equipo de desarrollo, casi siempre mucho mayor que el tier de
    hosting real del piloto, plan.md §2)."""

    def __init__(self, interval_s: float = 0.05):
        self._interval_s = interval_s
        self._process = psutil.Process()
        self._peak_rss_bytes = 0
        self._stop_event = threading.Event()
        self._thread: threading.Thread | None = None

    def _sample_loop(self) -> None:
        while not self._stop_event.is_set():
            rss = self._process.memory_info().rss
            self._peak_rss_bytes = max(self._peak_rss_bytes, rss)
            self._stop_event.wait(self._interval_s)

    def __enter__(self) -> "PeakRSSTracker":
        self._peak_rss_bytes = self._process.memory_info().rss
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._sample_loop, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc_info) -> None:
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)

    @property
    def peak_rss_mb(self) -> float:
        return round(self._peak_rss_bytes / (1024**2), 1)



def load_local_model(model_id: str = E5_MODEL_ID) -> tuple[SentenceTransformer, float]:
    t0 = time.perf_counter()
    model = SentenceTransformer(model_id)
    load_time_s = time.perf_counter() - t0
    return model, load_time_s


def embed_local(
    model: SentenceTransformer, texts: list[str], prefix: str, batch_size: int = 16
) -> tuple[list[list[float]], list[float]]:
    """prefix: 'query: ' o 'passage: '. Devuelve (vectores, latencia_ms_por_texto_por_lote)."""
    prefixed = [f"{prefix}{t}" for t in texts]
    latencies_ms: list[float] = []
    vectors: list[list[float]] = []
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i : i + batch_size]
        t0 = time.perf_counter()
        batch_vectors = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        latencies_ms.extend([elapsed_ms / len(batch)] * len(batch))
        vectors.extend(v.tolist() for v in batch_vectors)
    return vectors, latencies_ms


# PeakRSSTracker envuelve carga + embedding para capturar el pico real de RAM del
# proceso -- ese numero (no environment_info["ram_total_gb"]) es el que va en la
# fila "6. ¿corre en el servidor del piloto?" de la Sección 6.
with PeakRSSTracker() as ram_tracker_local:
    local_model, local_load_time_s = load_local_model()
    corpus_vectors_local, corpus_latencies_local = embed_local(local_model, corpus_texts, prefix="passage: ")
    query_vectors_local, query_latencies_local = embed_local(
        local_model, [q["query"] for q in golden_queries], prefix="query: "
    )
local_peak_ram_mb = ram_tracker_local.peak_rss_mb
print(f"Pico de RAM real (RSS) durante carga + embedding local: {local_peak_ram_mb} MB")

## 4. Candidato gestionado — `gemini-embedding-2`

Llamada REST directa (`embedContentConfig.taskType` / `outputDimensionality`, verificado en `https://ai.google.dev/api/embeddings` el 2026-07-09) en vez del wrapper `langchain-google-genai` instalado en el proyecto: la versión actual de ese wrapper no expone `output_dimensionality`, y el criterio 3/9/10 de research.md §1 exige comparar dimensiones distintas del mismo modelo (128–3072, Matryoshka). Precio verificado el 2026-07-09 en `https://ai.google.dev/gemini-api/docs/pricing`: USD 0,20 / 1M tokens de entrada (tier pagado); tier gratuito sin costo pero con límites de tasa más bajos — **vuelve a verificar el precio vigente antes de usarlo para una decisión de presupuesto real**, los proveedores cambian precios sin aviso.

In [18]:
import os
from pathlib import Path

import httpx
from dotenv import dotenv_values, load_dotenv

GEMINI_MODEL_ID = "gemini-embedding-2"
GEMINI_API_BASE = "https://generativelanguage.googleapis.com/v1beta"
GEMINI_BATCH_MAX = 100  # limite documentado de batchEmbedContents; confirmar si Google lo cambia
GEMINI_PRICE_PER_1M_TOKENS_USD = 0.20  # tier pagado, verificar vigencia antes de decidir
GEMINI_ENV_PATHS = [Path("../backend/.env"), Path("../.env.local"), Path("../.env")]

for env_path in GEMINI_ENV_PATHS:
    if env_path.exists():
        load_dotenv(env_path, override=False)


def _gemini_api_key() -> str:
    api_key = os.environ.get("GOOGLE_API_KEY")
    if api_key:
        return api_key
    for env_path in GEMINI_ENV_PATHS:
        if not env_path.exists():
            continue
        value = dotenv_values(env_path).get("GOOGLE_API_KEY")
        if value:
            return value
    raise RuntimeError("Falta GOOGLE_API_KEY en el entorno o en backend/.env")


def _gemini_headers() -> dict[str, str]:
    return {"x-goog-api-key": _gemini_api_key(), "Content-Type": "application/json"}


def embed_gemini(
    texts: list[str],
    task_type: str,
    output_dimensionality: int,
    model_id: str = GEMINI_MODEL_ID,
    batch_size: int = GEMINI_BATCH_MAX,
) -> tuple[list[list[float]], list[float], int]:
    """task_type: 'RETRIEVAL_QUERY' o 'RETRIEVAL_DOCUMENT'.
    Devuelve (vectores, latencia_ms_por_texto_por_lote, tokens_totales_reales)."""
    url = f"{GEMINI_API_BASE}/models/{model_id}:batchEmbedContents"
    vectors: list[list[float]] = []
    latencies_ms: list[float] = []
    total_tokens = 0
    with httpx.Client(timeout=30.0) as client:
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            body = {
                "requests": [
                    {
                        "model": f"models/{model_id}",
                        "content": {"parts": [{"text": text}]},
                        "embedContentConfig": {
                            "taskType": task_type,
                            "outputDimensionality": output_dimensionality,
                        },
                    }
                    for text in batch
                ]
            }
            t0 = time.perf_counter()
            response = client.post(url, headers=_gemini_headers(), json=body)
            response.raise_for_status()
            elapsed_ms = (time.perf_counter() - t0) * 1000
            payload = response.json()
            latencies_ms.extend([elapsed_ms / len(batch)] * len(batch))
            vectors.extend(item["values"] for item in payload["embeddings"])
            total_tokens += payload.get("usageMetadata", {}).get("promptTokenCount", 0)
    return vectors, latencies_ms, total_tokens


def costo_estimado_usd(n_tokens: int, precio_por_millon: float = GEMINI_PRICE_PER_1M_TOKENS_USD) -> float:
    return (n_tokens / 1_000_000) * precio_por_millon


# Descomentar para correr el candidato gestionado (requiere GOOGLE_API_KEY en el entorno o en backend/.env).
GEMINI_OUTPUT_DIM = 768  # <= 2000: compatible con vector(<DIM>) sin pasar a halfvec (research.md §1)
corpus_vectors_gemini, corpus_latencies_gemini, corpus_tokens_gemini = embed_gemini(
    corpus_texts, task_type="RETRIEVAL_DOCUMENT", output_dimensionality=GEMINI_OUTPUT_DIM
)
query_vectors_gemini, query_latencies_gemini, query_tokens_gemini = embed_gemini(
    [q["query"] for q in golden_queries], task_type="RETRIEVAL_QUERY", output_dimensionality=GEMINI_OUTPUT_DIM
)

## 5. Recall@10 y latencia p50/p95

Búsqueda por similitud coseno pura en memoria (sin pgvector): el objetivo de este notebook es comparar candidatos antes de decidir, no operar el índice real.

In [20]:
import numpy as np


def cosine_topk(query_vec: list[float], corpus_vecs: list[list[float]], k: int = 10) -> list[int]:
    q = np.array(query_vec)
    c = np.array(corpus_vecs)
    q = q / np.linalg.norm(q)
    c = c / np.linalg.norm(c, axis=1, keepdims=True)
    scores = c @ q
    return list(np.argsort(-scores)[:k])


def recall_at_k(
    queries: list[dict],
    query_vectors: list[list[float]],
    ids: list[str],
    corpus_vectors: list[list[float]],
    k: int = 10,
) -> float:
    hits = 0
    for q, qvec in zip(queries, query_vectors, strict=True):
        top_idx = cosine_topk(qvec, corpus_vectors, k=k)
        retrieved_ids = {ids[i] for i in top_idx}
        if retrieved_ids & set(q["expected_dataset_ids"]):
            hits += 1
    return hits / len(queries)


def recall_at_k_territorial(
    queries: list[dict],
    query_vectors: list[list[float]],
    ids: list[str],
    corpus_vectors: list[list[float]],
    k: int = 10,
) -> float:
    idx_territorial = [i for i, q in enumerate(queries) if q["territorial"]]
    subset_queries = [queries[i] for i in idx_territorial]
    subset_vectors = [query_vectors[i] for i in idx_territorial]
    return recall_at_k(subset_queries, subset_vectors, ids, corpus_vectors, k=k)


def p50_p95(latencies_ms: list[float]) -> tuple[float, float]:
    arr = np.array(latencies_ms)
    return float(np.percentile(arr, 50)), float(np.percentile(arr, 95))


# Ejemplo de uso una vez corridas las celdas 3 y 4 (descomentadas):
recall_local = recall_at_k(golden_queries, query_vectors_local, corpus_ids, corpus_vectors_local)
recall_local_territorial = recall_at_k_territorial(golden_queries, query_vectors_local, corpus_ids, corpus_vectors_local)
recall_gemini = recall_at_k(golden_queries, query_vectors_gemini, corpus_ids, corpus_vectors_gemini)
recall_gemini_territorial = recall_at_k_territorial(golden_queries, query_vectors_gemini, corpus_ids, corpus_vectors_gemini)

## 6. Tamaño del índice y tabla comparativa (10 criterios de research.md §1)

Tamaño estimado del índice: `n_datasets_totales_del_catalogo × dimensión × 4 bytes` (float32). Usa el conteo real del catálogo completo (T-201, ~8.400 datasets a la fecha de este PR — confirma con `SELECT count(*) FROM catalog_datasets WHERE api_active;`), no el tamaño de la muestra congelada.

In [ ]:
import pandas as pd

# Conteo real mas reciente documentado en tasks.md (T-201A, tercera ronda de
# auditoria de publicadores): 8.416 datasets activos en el catalogo COMPLETO real
# (NO la muestra congelada de 200 en notebooks/fixtures/catalog_sample.json --
# usar ese tamano de muestra aqui da un indice/costo de reindexacion ~42x mas
# chico que el real). Si ya tienes el catalogo completo cargado en tu propia
# base, reemplaza por tu conteo real:
#   docker compose exec db psql -U usuario -d cuestion_de_datos -c \
#     "SELECT count(*) FROM catalog_datasets WHERE api_active;"
CATALOG_TOTAL_DATASETS_ACTIVE = 8416


def tamano_indice_mb(n_datasets: int, dim: int) -> float:
    return round((n_datasets * dim * 4) / (1024**2), 2)


# Calcular resultados y rellenar la tabla con los valores medidos.
E5_DIM = 1024
GEMINI_DIM = 768
p50_local, p95_local = p50_p95(query_latencies_local)
p50_gemini, p95_gemini = p50_p95(query_latencies_gemini)

# Costo Gemini: tokens consumidos
costo_gemini_corpus_queries = costo_estimado_usd(corpus_tokens_gemini + query_tokens_gemini)
costo_reindex_gemini_est = costo_estimado_usd(
    (corpus_tokens_gemini + query_tokens_gemini) * (CATALOG_TOTAL_DATASETS_ACTIVE / len(corpus))
)

# local_peak_ram_mb viene de PeakRSSTracker en la Sección 3 (si esta linea falla con
# NameError, corre de nuevo la Sección 3 con la version actualizada -- ya mide el pico
# real de RAM en vez de un texto fijo "~32GB" que en realidad era la RAM TOTAL de la
# maquina, no el consumo real del modelo).
ram_row_local = f"Sí (CPU, pico medido: {local_peak_ram_mb} MB)"

comparacion = pd.DataFrame(
    {
        "criterio": [
            "1. recall@10 (todas las consultas)",
            "2. recall@10 (solo territoriales)",
            "3. dimensión del vector",
            "4. latencia p95 embed consulta (ms)",
            "5. costo estimado / 1.000 consultas (USD)",
            "5. costo estimado reindexación completa (USD)",
            "6. ¿corre en el servidor del piloto? (RAM/CPU medidos arriba vs. tier Railway/Render)",
            "7. dependencia de proveedor (lock-in)",
            "8. reproducibilidad (¿un tercero regenera el índice idéntico?)",
            "9. tamaño del índice resultante (MB, catálogo completo)",
            "10. compatibilidad pgvector (vector <=2000 / halfvec)",
        ],
        "intfloat/multilingual-e5-large (local)": [
            f"{recall_local:.2%}",
            f"{recall_local_territorial:.2%}",
            f"{E5_DIM} dim",
            f"{p95_local:.1f} ms",
            "$0.00 (local)",
            "$0.00 (local)",
            ram_row_local,
            "Ninguna (open model)",
            "Sí (determinístico, CPU)",
            f"{tamano_indice_mb(CATALOG_TOTAL_DATASETS_ACTIVE, E5_DIM)} MB",
            "vector(1024) ✓",
        ],
        "gemini-embedding-2 (gestionado)": [
            f"{recall_gemini:.2%}",
            f"{recall_gemini_territorial:.2%}",
            f"{GEMINI_DIM} dim",
            f"{p95_gemini:.1f} ms",
            f"${costo_estimado_usd(query_tokens_gemini * 10):.4f}",
            f"${costo_reindex_gemini_est:.2f}",
            "N/A (managed service)",
            "Alta (Google API dependency)",
            "No (closed model, API only)",
            f"{tamano_indice_mb(CATALOG_TOTAL_DATASETS_ACTIVE, GEMINI_DIM)} MB",
            "vector(768) ✓",
        ],
    }
)
print("\n=== RESULTADOS BENCHMARK T-205 ===\n")
print(f"Local (E5): recall@10={recall_local:.2%}, territorial={recall_local_territorial:.2%}, p95={p95_local:.1f}ms, RAM pico={local_peak_ram_mb} MB")
print(f"Gemini: recall@10={recall_gemini:.2%}, territorial={recall_gemini_territorial:.2%}, p95={p95_gemini:.1f}ms")
print(f"\nCosto Gemini corpus+queries: ${costo_gemini_corpus_queries:.4f}")
print(f"Costo Gemini reindex catalogo completo (~{CATALOG_TOTAL_DATASETS_ACTIVE} datasets, extrapolado desde tokens reales de la muestra): ${costo_reindex_gemini_est:.2f}")
print(f"\nTokens consumidos:")
print(f"  Corpus ({len(corpus)} datasets): {corpus_tokens_gemini} tokens")
print(f"  Queries ({len(golden_queries)} queries): {query_tokens_gemini} tokens")
print(f"  Total: {corpus_tokens_gemini + query_tokens_gemini} tokens\n")
print("Tabla comparativa:")
comparacion

## 7. Decisión razonada — A COMPLETAR Y FIRMAR POR EL RESPONSABLE DEL PROYECTO

`research.md` §1, procedimiento (f): *"decisión razonada firmada por el responsable del proyecto"*. Esta celda es una plantilla vacía a propósito — la decisión y su justificación las escribe Camilo con los resultados reales de arriba, no este notebook.

**Modelo elegido:** `gemini-embedding-2` (Google, gestionado)

**Dimensión elegida:** 768

**Tipo de columna pgvector (`vector` / `halfvec`):** `vector(768)` con `vector_cosine_ops` (768 <= 2000, no requiere `halfvec`)

**Justificación (referencia explícita a los 10 criterios medidos arriba):** El benchmark (research.md §1, 10 criterios) empata en recall@10 (94,12% general y 90,91% territorial para ambos candidatos sobre las 34 consultas anotadas a mano, 11 territoriales) -- ese criterio no discrimina. gemini-embedding-2 gana en latencia real medida (p95 41,8 ms vs 120,2 ms del local; ambos muy por debajo del presupuesto RNF-001 de 1 s) y, sobre todo, en costo de infraestructura real: el pico de RAM medido del candidato local (2.430,7 MB, `PeakRSSTracker`) excede el tier gratuito de Railway (0,5 GB) y, como RF-302 exige embeber la consulta "en caliente" en el mismo proceso del backend, esa RAM queda reservada de forma permanente dentro del tier Hobby (~USD 5/mes, plan.md §2) sin importar el volumen real de uso. El costo de la API gestionada es marginal al volumen del piloto (spec.md SUP-02, ~500 investigaciones/mes): $0,0092 por el corpus completo (200 datasets) + las 34 consultas de prueba (45.818 tokens reales), y $0,39 para una reindexación completa del catálogo activo (~8.416 datasets) -- un evento raro, no recurrente. Se acepta conscientemente perder los criterios 7 y 8 (dependencia de proveedor y reproducibilidad, donde el local es superior por tener pesos abiertos y ser determinístico): el proyecto ya tiene precedente de capa multi-proveedor para el LLM (RF-206) y research.md ya documenta que cambiar de modelo de embeddings exige regenerar el índice completo -- un costo conocido y acotado (~$0,39 al tamaño actual del catálogo), no catastrófico. Ambas dimensiones evaluadas (1024 local, 768 gestionado) caben en `vector(<=2000)` sin necesitar `halfvec`; se elige 768 (valor recomendado por Google para gemini-embedding-2 vía Matryoshka Representation Learning) por dar el índice más liviano (24,66 MB vs 32,88 MB sobre el catálogo completo) sin sacrificar recall frente a la dimensión nativa de 1024.

**Firma y fecha:** Juan Camilo Grajales B. — 2026-07-09

---

**Después de firmar, propaga la decisión en el mismo PR (`notebooks/README.md` tiene el detalle):**
- [x] `specs/001-cuestion-de-datos-v2/research.md` §1 — decisión + evidencia
- [x] `specs/001-cuestion-de-datos-v2/data-model.md` — reemplazar `<DIM>` en `catalog_embeddings`
- [x] `specs/001-cuestion-de-datos-v2/plan.md` — stack de embeddings, ya no "DECISIÓN PENDIENTE"
- [x] `backend/.env.example` y `quickstart.md` — `EMBEDDING_MODEL`
- [x] `backend/pyproject.toml` — dependencia definitiva de runtime (fuera de `benchmark-embeddings`)
- [x] T-104B queda desbloqueada en `tasks.md`